In [1]:
# import libraries

import pandas as pd
import numpy as np
import ast
import os
from google.colab import files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# Load and merge dataset
movies = pd.read_csv('/content/tmdb_5000_movies.csv')
credits = pd.read_csv('/content/tmdb_5000_credits.csv')

movies = movies.merge(credits, left_on='id', right_on='movie_id', suffixes=('', '_c'))
movies = movies[['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.dropna(inplace=True)
movies.reset_index(drop=True, inplace=True)

print(movies.shape)


(4800, 7)


In [4]:
# Data Preprocessing
def names(text):
    return [i['name'] for i in ast.literal_eval(text)]

def top3_cast(text):
    return [i['name'] for i in ast.literal_eval(text)[:3]]

def director(text):
    return [i['name'] for i in ast.literal_eval(text) if i['job'] == 'Director']

movies['genres'] = movies['genres'].apply(names)
movies['keywords'] = movies['keywords'].apply(names)
movies['cast'] = movies['cast'].apply(top3_cast)
movies['crew'] = movies['crew'].apply(director)
movies['overview'] = movies['overview'].apply(lambda x: x.split())


In [5]:
# spaces Remove (Sam Worthington -> SamWorthington)
for col in ['genres', 'keywords', 'cast', 'crew']:
    movies[col] = movies[col].apply(lambda x: [i.replace(" ", "") for i in x])

movies['tags'] = (movies['overview'] + movies['genres'] + movies['keywords']
                  + movies['cast'] + movies['crew'])
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x).lower())

df = movies[['id', 'title', 'tags']]




In [6]:
# TF-IDF + cosine similarity
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
vectors = tfidf.fit_transform(df['tags'])
similarity = cosine_similarity(vectors)

print("Vectors shape:", vectors.shape)
print("Similarity shape:", similarity.shape)


Vectors shape: (4800, 5000)
Similarity shape: (4800, 4800)


In [7]:
# Recommend function
def recommend(movie, n=5):
    matches = df[df['title'].str.lower() == movie.lower()]
    if matches.empty:
        return "Movie not found in dataset."
    idx = matches.index[0]
    scores = sorted(enumerate(similarity[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    return [df.iloc[i]['title'] for i, _ in scores]



In [8]:
# Test
for m in ['Avatar', 'The Dark Knight Rises', 'Titanic']:
    print(f"\nMovie: {m}")
    result = recommend(m)
    if isinstance(result, str):
        print(result)
    else:
        for i, r in enumerate(result, 1):
            print(f"{i}. {r}")



Movie: Avatar
1. Aliens
2. Falcon Rising
3. Battle: Los Angeles
4. Apollo 18
5. Star Trek Into Darkness

Movie: The Dark Knight Rises
1. The Dark Knight
2. Batman Returns
3. Batman Begins
4. Batman Forever
5. Batman

Movie: Titanic
1. Ghost Ship
2. Poseidon
3. In the Heart of the Sea
4. Pirates of the Caribbean: On Stranger Tides
5. Triangle


In [9]:
# input test
print("Enter movie name")
name = input("\nMovie ka naam likho: ")
result = recommend(name, n=10)
if isinstance(result, str):
    print(result)
else:
    for i, r in enumerate(result, 1):
        print(f"{i}. {r}")

Enter movie name

Movie ka naam likho: titanic
1. Ghost Ship
2. Poseidon
3. In the Heart of the Sea
4. Pirates of the Caribbean: On Stranger Tides
5. Triangle
6. The Black Hole
7. Captain Phillips
8. 20,000 Leagues Under the Sea
9. Supernova
10. The Bounty


In [10]:
import pickle
pickle.dump(df, open('movies.pkl', 'wb'))
pickle.dump(similarity.astype('float32'), open('similarity.pkl', 'wb'))